<a href="https://colab.research.google.com/github/Amoyeola/jupyter-exploration/blob/main/L11_Amoye_Peter_ITAI1378.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13 Lab: Deploying a Computer Vision Model as an API

Welcome to the final and most practical lab of the course! So far, you have learned how to build and train computer vision models. But how do you make your model useful to others? In this lab, you will learn how to **deploy** a model as an **API (Application Programming Interface)**. This will allow other applications to use your model to make predictions.

**What you'll learn:**
- What it means to deploy a model.
- What an API is and why it's useful.
- How to use the FastAPI framework to create a simple API for your model.
- How to send a request to your API and get a prediction back.

## Learning Objectives

By the end of this lab, you will be able to:

- **Explain** the concept of model deployment and its importance.
- **Create** a simple web API using FastAPI.
- **Integrate** a pre-trained computer vision model into a FastAPI application.
- **Deploy** a computer vision model as a local API.

## 1. Setup and Installation

We will need a few new libraries for this lab, including `fastapi` for creating the API and `uvicorn` for running it.


In [9]:
%pip install fastapi uvicorn python-multipart transformers torch ultralytics

## 2. Understanding Model Deployment and APIs

### What is Model Deployment?

**Model deployment** is the process of taking your trained model and making it available for use in a production environment. This means that other people and applications can send data to your model and get predictions back. It's the final step in the machine learning lifecycle.

### What is an API?

An **API (Application Programming Interface)** is a set of rules and protocols that allows different software applications to communicate with each other. In our case, we will create a web API that allows other applications to communicate with our model over the internet using standard HTTP requests.

### Why use FastAPI?

**FastAPI** is a modern, fast (high-performance) web framework for building APIs with Python. It is very easy to learn and use, and it automatically generates interactive API documentation, which is a huge plus!

## Part 1: Coded Demonstration

In this part, we will create a simple API that can classify an image using a pre-trained model from Hugging Face. Because we are in a notebook environment, we will write the code to a Python file and then run it.

In [3]:
# 1. Write the API code to a Python file

api_code = '''
from fastapi import FastAPI, File, UploadFile
from transformers import pipeline
from PIL import Image
import io

# Create the FastAPI app
app = FastAPI()

# Load the image classification pipeline
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

@app.get("/")
def read_root():
    return {"message": "Welcome to the Image Classification API!"}

@app.post("/classify/")
async def classify_image(file: UploadFile = File(...)):
    # Read the image file
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))

    # Get the prediction
    prediction = classifier(image)

    return {"filename": file.filename, "prediction": prediction}
'''

with open("main.py", "w") as f:
    f.write(api_code)

### 2. Run the API

Now, we will run the API using `uvicorn`. This will start a local web server that listens for requests. We will run this in the background so we can continue to use the notebook.

**Note:** You may need to stop and restart the kernel after running this cell to run the client code in the next step.

In [4]:
import subprocess
import time

# Start the server in the background
server_process = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

# Wait a moment for the server to start
time.sleep(5)

### 3. Test the API

Now that the API is running, let's send it an image and see what it predicts!

In [5]:
import requests
from PIL import Image
import io

# Download an image to test
image_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8&auto=format&fit=crop&w=1074&q=80"
response = requests.get(image_url)
image_bytes = io.BytesIO(response.content)

# Send the image to the API
files = {"file": ("dog.jpg", image_bytes, "image/jpeg")}
response = requests.post("http://localhost:8000/classify/", files=files)

# Print the prediction
print(response.json())

{'filename': 'dog.jpg', 'prediction': [{'label': 'beagle', 'score': 0.6593579649925232}, {'label': 'English foxhound', 'score': 0.26278847455978394}, {'label': 'Walker hound, Walker foxhound', 'score': 0.049733396619558334}, {'label': 'Brittany spaniel', 'score': 0.0036607717629522085}, {'label': 'EntleBucher', 'score': 0.003439259249716997}]}


### 4. Stop the API Server

It's important to stop the server process when you're done.

In [6]:
server_process.terminate()

## Part 2: Student Challenge

Your challenge is to create a new API that uses the YOLO object detection model from a previous lab. This will require you to modify the API to handle object detection instead of image classification.

**Your Task:**
1.  Create a new Python file called `object_detection_api.py`.
2.  In this file, create a FastAPI application that uses the `yolov8n.pt` model to detect objects in an image.
3.  The API should have an endpoint that accepts an image and returns a list of detected objects with their bounding boxes.
4.  Run your new API and test it with an image.

In [10]:
# --- ENTER YOUR CODE HERE ---

# 1. Write your object detection API code to a file

object_detection_api_code = '''
from fastapi import FastAPI, File, UploadFile
from ultralytics import YOLO
from PIL import Image
import io

app = FastAPI()

# Load the pretrained YOLO object-detection model
model = YOLO("yolov8n.pt")

@app.get("/")
def read_root():
    return {
        "message": "Welcome to the YOLO Object Detection API!"
    }

@app.post("/detect/")
async def detect_objects(file: UploadFile = File(...)):
    # Read and convert the uploaded image
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Run object detection
    results = model.predict(
        source=image,
        verbose=False
    )

    detections = []

    # Extract labels, confidence scores, and bounding boxes
    for box in results[0].boxes:
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())
        coordinates = box.xyxy[0].tolist()

        detections.append({
            "class_name": model.names[class_id],
            "confidence": round(confidence, 4),
            "bounding_box": {
                "x1": round(coordinates[0], 2),
                "y1": round(coordinates[1], 2),
                "x2": round(coordinates[2], 2),
                "y2": round(coordinates[3], 2)
            }
        })

    return {
        "filename": file.filename,
        "number_of_detections": len(detections),
        "detections": detections
    }
'''

with open("object_detection_api.py", "w") as f:
    f.write(object_detection_api_code)

print("object_detection_api.py created successfully.")

# 2. Run your API

import subprocess
import time

object_server_process = subprocess.Popen(
    [
        "uvicorn",
        "object_detection_api:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8001"
    ]
)

# Allow time for YOLO to load
time.sleep(20)

print("YOLO object-detection API server started.")

# 3. Test your API
import requests
import io
import json

test_image_url = "https://ultralytics.com/images/bus.jpg"
image_response = requests.get(test_image_url)
image_response.raise_for_status()

test_image_bytes = io.BytesIO(image_response.content)

files = {
    "file": (
        "bus.jpg",
        test_image_bytes,
        "image/jpeg"
    )
}

api_response = requests.post(
    "http://localhost:8001/detect/",
    files=files
)

print("Status code:", api_response.status_code)
print(json.dumps(api_response.json(), indent=2))

# 4. Stop your API
object_server_process.terminate()
object_server_process.wait()

print("YOLO object-detection API server stopped.")

object_detection_api.py created successfully.
YOLO object-detection API server started.
Status code: 200
{
  "filename": "bus.jpg",
  "number_of_detections": 6,
  "detections": [
    {
      "class_name": "bus",
      "confidence": 0.8734,
      "bounding_box": {
        "x1": 22.87,
        "y1": 231.28,
        "x2": 805.0,
        "y2": 756.84
      }
    },
    {
      "class_name": "person",
      "confidence": 0.8657,
      "bounding_box": {
        "x1": 48.55,
        "y1": 398.55,
        "x2": 245.35,
        "y2": 902.7
      }
    },
    {
      "class_name": "person",
      "confidence": 0.8528,
      "bounding_box": {
        "x1": 669.47,
        "y1": 392.19,
        "x2": 809.72,
        "y2": 877.04
      }
    },
    {
      "class_name": "person",
      "confidence": 0.8252,
      "bounding_box": {
        "x1": 221.52,
        "y1": 405.8,
        "x2": 344.97,
        "y2": 857.54
      }
    },
    {
      "class_name": "person",
      "confidence": 0.2611,
     

## Reflective Questions

Please answer the following questions in a new Markdown cell below.

1.  Why is it better to deploy a model as an API instead of just having it as a script on your computer? What are the advantages?

- Deploying a model as an API makes it available to other programs without requiring them to contain or understand the model’s internal code. Different applications can send images to one centralized model and receive standardized predictions in JSON format. This makes the model easier to reuse, maintain, update, and integrate with websites, mobile applications, or business systems.

2.  FastAPI automatically creates documentation for your API. How could this be useful for a team of developers working on a project?

- FastAPI’s automatic documentation helps developers understand the available endpoints, required inputs, and expected outputs. Team members can test the API directly from the documentation page without writing a separate client first. This improves communication, reduces integration mistakes, and makes troubleshooting easier.


3.  Our API was deployed locally on our computer. What are some of the challenges you would face if you wanted to deploy this API to the cloud so that anyone in the world could use it? (Think about scalability, cost, security, etc.)

- Cloud deployment would introduce challenges involving scalability, cost, security, and reliability. The system would need enough CPU or GPU resources to handle multiple requests, secure file uploads, control access, protect user data, and monitor failures. The organization would also need to manage model-loading time, storage, network traffic, software dependencies, and cloud-service expenses.

4.  What is MLOps and how does it relate to model deployment? Why is it an important field in AI?

- MLOps is the practice of managing the complete machine-learning lifecycle using reliable development and operations processes. It supports model deployment through version control, testing, monitoring, automated updates, and performance tracking. MLOps is important because a model can lose accuracy or fail after deployment if changes in data, software, or infrastructure are not detected and managed.

## Submission Instructions

1.  Complete the **Student Challenge** section with your object detection API.
2.  Answer the **Reflective Questions** in a new Markdown cell.
3.  Save your completed notebook (`.ipynb` file) and submit it.